In [64]:
import os
import json
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

from transformers import CLIPVisionModel, AutoProcessor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [55]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7845.72it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.embeddings.position_embedding.weight              | UNEXP

In [56]:
class CustomData(Dataset):
    def __init__(self, json_file):
        with open(json_file, "r") as f:
            self.data = json.load(f)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        image = Image.open(item["image_path"]).convert("RGB")
        label = item["label"]

        inputs = processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)

        return pixel_values, label

In [57]:
dataset = CustomData("mapping.json")

indexes = list(range(len(dataset)))

train_idx, test_idx = train_test_split(indexes, test_size=0.2, random_state=42)

train_dataset = Subset(dataset, train_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
class CLIPClassifier(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model

        self.classifier = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 2)
        )

    def forward(self, pixel_values):
        outputs = self.clip(pixel_values=pixel_values)
        x = outputs.pooler_output
        return self.classifier(x)

In [ ]:
model_clf = CLIPClassifier(clip).to(device)


for p in model_clf.clip.parameters():
    p.requires_grad = False

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0]).to(device))
optimizer = torch.optim.AdamW(model_clf.parameters(), lr=5e-5)

In [60]:
for epoch in range(10):
    model_clf.train()
    total_loss = 0

    for batch_image, batch_label in train_loader:

        batch_image = batch_image.to(device)
        batch_label = batch_label.to(device)

        outputs = model_clf(batch_image)
        loss = criterion(outputs, batch_label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} - loss: {total_loss:.4f}")

Epoch 0 - loss: 11.3138
Epoch 1 - loss: 10.1833
Epoch 2 - loss: 7.0588
Epoch 3 - loss: 6.6601
Epoch 4 - loss: 7.5081
Epoch 5 - loss: 6.7660
Epoch 6 - loss: 5.2586
Epoch 7 - loss: 4.1911
Epoch 8 - loss: 4.1666
Epoch 9 - loss: 3.3336


In [61]:
model_clf.eval()

y_test = []
y_score = []

with torch.no_grad():
    for batch_image, batch_label in test_loader:

        batch_image = batch_image.to(device)

        outputs = model_clf(batch_image)
        probs = torch.softmax(outputs, dim=1)

        y_score.extend(probs[:, 1].cpu().numpy())
        y_test.extend(batch_label.cpu().numpy())

In [62]:
thresholds = np.arange(0.1, 0.9, 0.05)

best_t = 0
best_f1 = 0

for t in thresholds:

    y_pred = (np.array(y_score) > t).astype(int)

    f1 = f1_score(y_test, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

Best threshold: 0.5000000000000001
Best F1: 0.8070175438596491


In [63]:
y_pred = (np.array(y_score) > best_t).astype(int)

print("Threshold:", best_t)
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))

print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))

print("\nReport:\n", classification_report(y_test, y_pred))

Threshold: 0.5000000000000001
Precision: 0.8846153846153846
Recall: 0.7419354838709677

Confusion matrix:
 [[43  3]
 [ 8 23]]

Report:
               precision    recall  f1-score   support

           0       0.84      0.93      0.89        46
           1       0.88      0.74      0.81        31

    accuracy                           0.86        77
   macro avg       0.86      0.84      0.85        77
weighted avg       0.86      0.86      0.85        77

